In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/twcs.csv")

In [2]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nFirst 5 rows:")
print(df.head())

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
print(df.describe())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nUnique values in inbound:")
print(df["inbound"].unique())

print("\nCount of each inbound value:")
print(df["inbound"].value_counts())

Shape: (2811774, 7)

Columns:
Index(['tweet_id', 'author_id', 'inbound', 'created_at', 'text',
       'response_tweet_id', 'in_response_to_tweet_id'],
      dtype='object')

First 5 rows:
   tweet_id   author_id  inbound                      created_at  \
0         1  sprintcare    False  Tue Oct 31 22:10:47 +0000 2017   
1         2      115712     True  Tue Oct 31 22:11:45 +0000 2017   
2         3      115712     True  Tue Oct 31 22:08:27 +0000 2017   
3         4  sprintcare    False  Tue Oct 31 21:54:49 +0000 2017   
4         5      115712     True  Tue Oct 31 21:49:35 +0000 2017   

                                                text response_tweet_id  \
0  @115712 I understand. I would like to assist y...                 2   
1      @sprintcare and how do you propose we do that               NaN   
2  @sprintcare I have sent several private messag...                 1   
3  @115712 Please send us a Private Message so th...                 3   
4                                

In [3]:
# Calculate author-level statistics

author_stats = df.groupby("author_id").agg(
    inbound_tweets=("inbound", "sum"),
    total_tweets=("tweet_id", "count")
)

author_stats["outbound_tweets"] = (
    author_stats["total_tweets"] -
    author_stats["inbound_tweets"]
)

author_stats = author_stats[
    ["inbound_tweets", "outbound_tweets", "total_tweets"]
]

print("\nAuthor-level statistics:")
print(author_stats.head())

# Authors with at least one outbound tweet
authors_with_outbound = author_stats[
    author_stats["outbound_tweets"] > 0
]

print("\nAuthors with at least one outbound tweet:")
print(authors_with_outbound.shape)

# Top authors by outbound tweet count
top_outbound = authors_with_outbound.sort_values(
    "outbound_tweets",
    ascending=False
)

print(top_outbound.head(20))


Author-level statistics:
           inbound_tweets  outbound_tweets  total_tweets
author_id                                               
10026                   3                0             3
100363                  1                0             1
10103                   2                0             2
10221                   2                0             2
10286                   1                0             1

Authors with at least one outbound tweet:
(108, 3)
                 inbound_tweets  outbound_tweets  total_tweets
author_id                                                     
AmazonHelp                    0           169840        169840
AppleSupport                  0           106860        106860
Uber_Support                  0            56270         56270
SpotifyCares                  0            43265         43265
Delta                         0            42253         42253
Tesco                         0            38573         38573
AmericanAir        

In [4]:
# Select only outbound tweets
outbound = df[df["inbound"] == False]

# Connect each outbound tweet to the tweet it is responding to
outbound_customer = outbound.merge(
    df[["tweet_id", "author_id"]],
    left_on="in_response_to_tweet_id",
    right_on="tweet_id",
    suffixes=("_brand", "_customer")
)

# Count distinct customers for each brand
customer_counts = (
    outbound_customer
    .groupby("author_id_brand")["author_id_customer"]
    .nunique()
    .reset_index(name="distinct_customers")
)

# Sort by number of customers
customer_counts = customer_counts.sort_values(
    "distinct_customers",
    ascending=False
)

# Display top authors
customer_counts.head(20)

,author_id_brand,distinct_customers
10,AppleSupport,76366
8,AmazonHelp,71049
85,Uber_Support,38300
77,SpotifyCares,27794
40,Delta,22331
99,comcastcares,21824
9,AmericanAir,21686
78,TMobileHelp,19943
76,SouthwestAir,19713
26,Ask_Spectrum,17214
